In [5]:
%matplotlib inline
import os
import sys
from pathlib import Path
from urllib.parse import quote_plus

import pandas as pd


from dotenv import load_dotenv
from sqlalchemy import create_engine

_NOTEBOOK_DIR = Path.cwd()
_PROJECT_ROOT = _NOTEBOOK_DIR
for _ in range(5):
    if (_PROJECT_ROOT / ".env").exists():
        break
    _PROJECT_ROOT = _PROJECT_ROOT.parent

# Ensure Training dir is on path so data_prep and data_prep_batters can be imported
sys.path.insert(0, str(_NOTEBOOK_DIR))

def _load_env():
    load_dotenv(_PROJECT_ROOT / ".env")

def _db_url():
    host = os.getenv("PGHOST")
    port = os.getenv("PGPORT")
    user = os.getenv("PGUSER")
    password = os.getenv("PGPASSWORD")
    dbname = os.getenv("PGDATABASE")
    pw = quote_plus(password) if password else ""
    return f"postgresql://{user}:{pw}@{host}:{port}/{dbname}"

_load_env()
engine = create_engine(_db_url())


In [6]:
from sklearn.metrics import log_loss, accuracy_score, mean_squared_error

import numpy as np
import optuna
import xgboost as xgb

from data_prep import load_pitcher_simulator_data, prepare_pitcher_features
from temporal_split import temporal_train_val_test

# Deciding counts (3-2, 0-2, 1-2, 2-2, 3-1) for count-aware sample weighting
DECIDING_COUNTS = {(0, 2), (1, 2), (2, 2), (3, 1), (3, 2)}
def is_deciding_count(balls: int, strikes: int) -> bool:
    return (int(balls), int(strikes)) in DECIDING_COUNTS

print("imported modules")

imported modules


In [7]:
df_raw = load_pitcher_simulator_data(engine)
train_df, val_df, test_df, split_meta = temporal_train_val_test(df_raw)
print("Temporal split:", split_meta)

X_train, y_train_dict, pitcher_enc = prepare_pitcher_features(train_df)
X_val, y_val_dict, _ = prepare_pitcher_features(val_df, encoders=pitcher_enc)
X_test, y_test_dict, _ = prepare_pitcher_features(test_df, encoders=pitcher_enc)

print(f"Loaded {len(df_raw)} pitches")
print(f"Features: {list(X_train.columns)}")
print(f"Targets: {list(y_train_dict.keys())}")
print(f"\nTrain: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")

Temporal split: {'years_present': [2023, 2024, 2025], 'split_kind': 'multi_season', 'val_year': 2024, 'test_year': 2025, 'n_train': 713439, 'n_val': 703603, 'n_test': 707385}
Loaded 2124427 pitches
Features: ['pitcher', 'p_throws', 'stand', 'balls', 'strikes', 'is_pitcher_count', 'is_batter_count', 'inning', 'inning_topbot', 'outs_when_up', 'at_bat_number', 'pitch_number', 'previous_pitch_type', 'previous_release_speed', 'pitcher_pitches_this_game', 'pitcher_pitches_this_inning', 'game_date', 'home_team', 'away_team', 'game_type', 'pitcher_career_ip', 'pitcher_career_era', 'pitcher_career_so', 'pitcher_career_bb', 'pitcher_career_h', 'pitcher_career_er', 'pitcher_career_hr', 'pitcher_career_bfp', 'pitcher_career_ipouts']
Targets: ['pitch_type', 'plate_x', 'plate_z', 'release_speed', 'release_spin_rate']

Train: 713439 | Val: 703603 | Test: 707385


In [8]:
import json

repertoire_by_pitcher = (
    pd.DataFrame({"pitcher": X_train["pitcher"], "pitch_type": y_train_dict["pitch_type"]})
    .groupby("pitcher")["pitch_type"]
    .apply(lambda s: sorted(s.unique().tolist()))
    .to_dict()
)
# JSON keys must be strings
pitcher_repertoire_json = {str(int(k)): v for k, v in repertoire_by_pitcher.items()}
_SAVED = Path(_NOTEBOOK_DIR) / "saved_models"
_SAVED.mkdir(parents=True, exist_ok=True)
with open(_SAVED / "pitcher_repertoire.json", "w") as f:
    json.dump(pitcher_repertoire_json, f, indent=2)
print(f"Saved pitcher_repertoire.json ({len(pitcher_repertoire_json)} pitchers)")

Saved pitcher_repertoire.json (863 pitchers)


In [9]:
from sklearn.utils.class_weight import compute_class_weight

codes_in_train = sorted(y_train_dict["pitch_type"].unique())
type_to_idx = {t: i for i, t in enumerate(codes_in_train)}
map_pitch_types = {i: t for t, i in type_to_idx.items()}

y_train_pt = y_train_dict["pitch_type"].map(type_to_idx)
y_val_pt = y_val_dict["pitch_type"].map(lambda t: type_to_idx.get(t, 0))
y_test_pt = y_test_dict["pitch_type"].map(lambda t: type_to_idx.get(t, 0))

classes_pt = np.unique(y_train_pt)
class_weights_pt = compute_class_weight(
    "balanced", classes=classes_pt, y=y_train_pt.to_numpy().ravel()
)
class_weight_per_sample = class_weights_pt[np.searchsorted(classes_pt, y_train_pt.to_numpy().ravel())]

# Count-awareness: upweight deciding counts (3-2, 0-2, 1-2, 2-2, 3-1) so model learns pitcher behavior in those counts
DECIDING_COUNT_WEIGHT = 1.5
is_deciding_train = X_train.apply(lambda r: is_deciding_count(int(r["balls"]), int(r["strikes"])), axis=1)
deciding_multiplier = np.where(is_deciding_train.values, DECIDING_COUNT_WEIGHT, 1.0)
sample_weight_pt = class_weight_per_sample * deciding_multiplier

# For regressors (same row order as X_train)
sample_weight_reg = deciding_multiplier.astype(np.float64)

X_train_pt = X_train
y_train_pt_oversampled = y_train_pt

feats_s1 = list(X_train.columns)

In [ ]:
# Hyperparameter tuning: Optuna minimizes log loss on the temporal validation set (no test leakage).
def tune_pitch_type(trial):
    n_estimators = trial.suggest_int("n_estimators", 15, 500)
    max_depth = trial.suggest_int("max_depth", 3, 12)
    learning_rate = trial.suggest_float("learning_rate", 0.01, 0.3)
    subsample = trial.suggest_float("subsample", 0.5, 1.0)
    colsample_bytree = trial.suggest_float("colsample_bytree", 0.5, 1.0)
    model = xgb.XGBClassifier(
        objective="multi:softprob", random_state=42,
        n_estimators=n_estimators, max_depth=max_depth,
        learning_rate=learning_rate, subsample=subsample, colsample_bytree=colsample_bytree,
    )
    model.fit(X_train_pt[feats_s1], y_train_pt_oversampled, sample_weight=sample_weight_pt)
    proba = model.predict_proba(X_val[feats_s1])
    # Pass labels so log_loss accepts when val set has fewer classes than train (e.g. 16 vs 17)
    return log_loss(y_val_pt, proba, labels=np.arange(len(codes_in_train)))

study_pt = optuna.create_study(direction="minimize")
study_pt.optimize(tune_pitch_type, n_trials=25)
print("Best log_loss:", study_pt.best_value, "| Best params:", study_pt.best_params)


[I 2026-04-23 02:31:22,433] A new study created in memory with name: no-name-64c30278-d45b-4608-8a94-dccda6be931c
[I 2026-04-23 02:33:02,338] Trial 0 finished with value: 1.7881256445687854 and parameters: {'n_estimators': 392, 'max_depth': 9, 'learning_rate': 0.2753865223726943, 'subsample': 0.6025171950112476, 'colsample_bytree': 0.6791253105991831}. Best is trial 0 with value: 1.7881256445687854.
[I 2026-04-23 02:34:09,463] Trial 1 finished with value: 1.5820781610134358 and parameters: {'n_estimators': 227, 'max_depth': 12, 'learning_rate': 0.05449831150505099, 'subsample': 0.6397980057151154, 'colsample_bytree': 0.742591906961064}. Best is trial 1 with value: 1.5820781610134358.
[I 2026-04-23 02:35:29,924] Trial 2 finished with value: 1.606696586460025 and parameters: {'n_estimators': 348, 'max_depth': 10, 'learning_rate': 0.09656258316435289, 'subsample': 0.9901669367159718, 'colsample_bytree': 0.5766197744684408}. Best is trial 1 with value: 1.5820781610134358.
[I 2026-04-23 02:

Best log_loss: 1.5820781610134358 | Best params: {'n_estimators': 227, 'max_depth': 12, 'learning_rate': 0.05449831150505099, 'subsample': 0.6397980057151154, 'colsample_bytree': 0.742591906961064}


In [11]:
# Addressing memorizing the majority / overfitting: we oversample rare pitch types (and cap
# common ones at the median count) so the model learns all types instead of memorizing noise
# or collapsing to the most frequent class
rs = np.random.RandomState(42)
n_classes = len(codes_in_train)
counts = y_train_pt.value_counts().reindex(range(n_classes), fill_value=0).values
target_per_class = int(np.median(counts[counts > 0]))  # avoid huge dataset
balanced_idx = []
for c in range(n_classes):
    idx_c = np.where(y_train_pt.values == c)[0]
    n_c = len(idx_c)
    if n_c == 0:
        continue
    if n_c >= target_per_class:
        chosen = rs.choice(idx_c, size=target_per_class, replace=False)
    else:
        chosen = rs.choice(idx_c, size=target_per_class, replace=True)
    balanced_idx.extend(chosen)
balanced_idx = np.array(balanced_idx)
rs.shuffle(balanced_idx)

X_train_pt = X_train.iloc[balanced_idx].reset_index(drop=True)
y_train_pt_oversampled = y_train_pt.iloc[balanced_idx].reset_index(drop=True)
# Align sample weights with oversampled rows (same length as X_train_pt)
sample_weight_pt = np.asarray(sample_weight_pt)[balanced_idx]
print(f"Pitch-type training: original {len(X_train)} -> oversampled {len(X_train_pt)} (target {target_per_class} per class)")

Pitch-type training: original 713439 -> oversampled 239408 (target 14963 per class)


In [12]:
xgb_pt = xgb.XGBClassifier(objective="multi:softprob", random_state=42, **study_pt.best_params)
xgb_pt.fit(X_train_pt[feats_s1], y_train_pt_oversampled, sample_weight=sample_weight_pt)

col_names = [map_pitch_types[i] for i in range(len(codes_in_train))]
proba_pt = xgb_pt.predict_proba(X_test[feats_s1])
pitch_type_pred = pd.Series(
    [col_names[i] for i in np.argmax(proba_pt, axis=1)],
    index=X_test.index,
)

_SAVED = Path(_NOTEBOOK_DIR) / "saved_models"
_SAVED.mkdir(parents=True, exist_ok=True)
xgb_pt.save_model(str(_SAVED / "pitcher_pitch_type.json"))
print("Saved pitcher_pitch_type.json")

Saved pitcher_pitch_type.json


In [13]:
X_train_s2 = X_train.copy()
X_train_s2["pitch_type_code"] = y_train_dict["pitch_type"].map(type_to_idx)

X_val_s2 = X_val.copy()
X_val_s2["pitch_type_code"] = y_val_dict["pitch_type"].map(lambda t: type_to_idx.get(t, 0))

X_test_s2 = X_test.copy()
X_test_s2["pitch_type_code"] = pitch_type_pred.map(lambda t: type_to_idx.get(t, 0))

feats_s2 = feats_s1 + ["pitch_type_code"]

In [14]:
# Hyperparameter tuning for the four regressors (plate_x, plate_z, release_speed, release_spin_rate)
# Optuna runs n_trials per target and we fit and save the model with the best params for each
def tune_reg(trial, target_name):
    n_estimators = trial.suggest_int("n_estimators", 15, 500)
    max_depth = trial.suggest_int("max_depth", 3, 12)
    learning_rate = trial.suggest_float("learning_rate", 0.01, 0.3)
    subsample = trial.suggest_float("subsample", 0.5, 1.0)
    colsample_bytree = trial.suggest_float("colsample_bytree", 0.5, 1.0)
    reg_alpha = trial.suggest_float("reg_alpha", 0.01, 10.0)
    reg_lambda = trial.suggest_float("reg_lambda", 0.1, 10.0)
    min_child_weight = trial.suggest_int("min_child_weight", 1, 10)
    model = xgb.XGBRegressor(objective="reg:squarederror", random_state=42,
        n_estimators=n_estimators, max_depth=max_depth,
        learning_rate=learning_rate, subsample=subsample, colsample_bytree=colsample_bytree,
        reg_alpha=reg_alpha, reg_lambda=reg_lambda, min_child_weight=min_child_weight)
    model.fit(X_train_s2[feats_s2], y_train_dict[target_name], sample_weight=sample_weight_reg)
    pred = model.predict(X_val_s2[feats_s2])
    return np.sqrt(mean_squared_error(y_val_dict[target_name], pred))  # RMSE on temporal val

reg_targets = ["plate_x", "plate_z", "release_speed", "release_spin_rate"]
reg_models = {}
for tgt in reg_targets:
    study = optuna.create_study(direction="minimize")
    study.optimize(lambda trial, t=tgt: tune_reg(trial, t), n_trials=10)
    model = xgb.XGBRegressor(objective="reg:squarederror", random_state=42, **study.best_params)
    model.fit(X_train_s2[feats_s2], y_train_dict[tgt], sample_weight=sample_weight_reg)
    model.save_model(str(_SAVED / f"pitcher_{tgt}.json"))
    reg_models[tgt] = model
    print(f"pitcher_{tgt}.json: best RMSE = {study.best_value:.4f}")

[I 2026-04-23 02:43:45,547] A new study created in memory with name: no-name-01dd30ca-31ab-4283-82d8-c2bc29f015e7
[I 2026-04-23 02:43:47,008] Trial 0 finished with value: 0.7659549087418019 and parameters: {'n_estimators': 126, 'max_depth': 6, 'learning_rate': 0.062242125502545514, 'subsample': 0.5426672521973125, 'colsample_bytree': 0.6582615706677939, 'reg_alpha': 5.063061997607249, 'reg_lambda': 6.065012839990127, 'min_child_weight': 8}. Best is trial 0 with value: 0.7659549087418019.
[I 2026-04-23 02:43:50,832] Trial 1 finished with value: 0.7788719616537864 and parameters: {'n_estimators': 359, 'max_depth': 7, 'learning_rate': 0.23248922698470237, 'subsample': 0.809101073509151, 'colsample_bytree': 0.9355949632723919, 'reg_alpha': 6.273232766666928, 'reg_lambda': 4.9510758198661025, 'min_child_weight': 9}. Best is trial 0 with value: 0.7659549087418019.
[I 2026-04-23 02:43:56,379] Trial 2 finished with value: 0.781263574406234 and parameters: {'n_estimators': 408, 'max_depth': 9, 

pitcher_plate_x.json: best RMSE = 0.7630


[I 2026-04-23 02:44:29,861] Trial 0 finished with value: 0.8383082255013676 and parameters: {'n_estimators': 191, 'max_depth': 5, 'learning_rate': 0.23173582576055918, 'subsample': 0.7846717768335252, 'colsample_bytree': 0.7303069461739979, 'reg_alpha': 5.384580461100007, 'reg_lambda': 5.195805472325433, 'min_child_weight': 7}. Best is trial 0 with value: 0.8383082255013676.
[I 2026-04-23 02:44:31,162] Trial 1 finished with value: 0.8408124962937424 and parameters: {'n_estimators': 209, 'max_depth': 3, 'learning_rate': 0.2062252080798761, 'subsample': 0.8955427971587597, 'colsample_bytree': 0.5569883108677326, 'reg_alpha': 7.002820296751914, 'reg_lambda': 6.781705789572749, 'min_child_weight': 10}. Best is trial 0 with value: 0.8383082255013676.
[I 2026-04-23 02:44:33,161] Trial 2 finished with value: 0.8400244460807972 and parameters: {'n_estimators': 262, 'max_depth': 4, 'learning_rate': 0.04869786311752928, 'subsample': 0.5209405900807393, 'colsample_bytree': 0.7763041986922795, 're

pitcher_plate_z.json: best RMSE = 0.8337


[I 2026-04-23 02:45:05,365] Trial 0 finished with value: 1.7376498774981792 and parameters: {'n_estimators': 351, 'max_depth': 8, 'learning_rate': 0.2401929425141363, 'subsample': 0.6497438901128185, 'colsample_bytree': 0.9094710968056483, 'reg_alpha': 6.2622364092405745, 'reg_lambda': 1.6952996430852634, 'min_child_weight': 4}. Best is trial 0 with value: 1.7376498774981792.
[I 2026-04-23 02:45:08,981] Trial 1 finished with value: 1.7204935392796865 and parameters: {'n_estimators': 133, 'max_depth': 12, 'learning_rate': 0.1274912309832817, 'subsample': 0.6787338602111357, 'colsample_bytree': 0.7647756478677031, 'reg_alpha': 6.279285671372, 'reg_lambda': 1.9545198819138827, 'min_child_weight': 5}. Best is trial 1 with value: 1.7204935392796865.
[I 2026-04-23 02:45:09,497] Trial 2 finished with value: 2.1844984640121945 and parameters: {'n_estimators': 47, 'max_depth': 4, 'learning_rate': 0.19145747858911366, 'subsample': 0.7357924874092324, 'colsample_bytree': 0.9356265984855421, 'reg_

pitcher_release_speed.json: best RMSE = 1.7205


[I 2026-04-23 02:45:31,084] Trial 0 finished with value: 157.45153962240343 and parameters: {'n_estimators': 264, 'max_depth': 9, 'learning_rate': 0.10552984161623181, 'subsample': 0.657037641523454, 'colsample_bytree': 0.7280993398890224, 'reg_alpha': 7.8602372589953395, 'reg_lambda': 9.350239437497319, 'min_child_weight': 4}. Best is trial 0 with value: 157.45153962240343.
[I 2026-04-23 02:45:36,312] Trial 1 finished with value: 159.31093329436348 and parameters: {'n_estimators': 305, 'max_depth': 10, 'learning_rate': 0.023060958611155966, 'subsample': 0.6516343438805581, 'colsample_bytree': 0.8135713396290671, 'reg_alpha': 7.835196143343357, 'reg_lambda': 3.094094224992147, 'min_child_weight': 7}. Best is trial 0 with value: 157.45153962240343.
[I 2026-04-23 02:45:40,301] Trial 2 finished with value: 160.058600395418 and parameters: {'n_estimators': 413, 'max_depth': 7, 'learning_rate': 0.07015449476125073, 'subsample': 0.9886676456917601, 'colsample_bytree': 0.8890393882640402, 're

pitcher_release_spin_rate.json: best RMSE = 156.6220


In [15]:
#compute how well the pitch type model and the four regressors do on the test set
acc_pt = accuracy_score(y_test_pt, xgb_pt.predict(X_test[feats_s1]))
proba_pt_test = xgb_pt.predict_proba(X_test[feats_s1])
y_test_onehot = np.zeros_like(proba_pt_test)
for i, pt in enumerate(y_test_pt):
    if 0 <= pt < proba_pt_test.shape[1]:
        y_test_onehot[i, int(pt)] = 1
loss_pt = log_loss(y_test_onehot, proba_pt_test)
print("Pitch type: accuracy =", f"{acc_pt:.4f}", "| log loss =", f"{loss_pt:.4f}")

for tgt in reg_targets:
    pred = reg_models[tgt].predict(X_test_s2[feats_s2])
    rmse = np.sqrt(mean_squared_error(y_test_dict[tgt], pred))
    mae = np.abs(y_test_dict[tgt].values - pred).mean()
    print(f"{tgt}: RMSE = {rmse:.4f} | MAE = {mae:.4f}")

Pitch type: accuracy = 0.2336 | log loss = 2.8606
plate_x: RMSE = 0.8335 | MAE = 0.6589
plate_z: RMSE = 1.0550 | MAE = 0.8258
release_speed: RMSE = 8.0554 | MAE = 6.3953
release_spin_rate: RMSE = 455.4342 | MAE = 325.4927


In [16]:
# Monte Carlo at inference + avoiding memorizing noise: compute the std of prediction errors
# per pitch type and save it. The app then adds random noise with this scale when predicting,
# so the simulator outputs a distribution of plausible outcomes instead of a single deterministic
# prediction that would overfit to training noise
import json

pt_train = y_train_dict["pitch_type"]  # pitch type strings
residual_stds = {}
DEFAULT_STDS = {"plate_x": 0.5, "plate_z": 0.5, "release_speed": 1.5, "release_spin_rate": 200}
MIN_SAMPLES = 10

for tgt in reg_targets:
    pred = reg_models[tgt].predict(X_train_s2[feats_s2])
    res = y_train_dict[tgt].values - pred
    df_res = pd.DataFrame({"pt": pt_train.values, "res": res})
    by_pt = df_res.groupby("pt")["res"].agg(["std", "count"])
    global_std = df_res["res"].std()
    std_by_pt = {}
    for pt in by_pt.index:
        row = by_pt.loc[pt]
        std_by_pt[pt] = float(row["std"]) if row["count"] >= MIN_SAMPLES and pd.notna(row["std"]) else float(global_std)
    residual_stds[tgt] = std_by_pt

with open(_SAVED / "residual_stds.json", "w") as f:
    json.dump(residual_stds, f, indent=2)
print("Saved residual_stds.json")

Saved residual_stds.json


In [17]:
#for each pitcher and each pitch type compute average location and save so the app can nudge predictions toward it
MIN_PITCHES_FOR_MEANS = 20
df_loc = pd.DataFrame({
    "pitcher": X_train["pitcher"],
    "pitch_type": y_train_dict["pitch_type"],
    "plate_x": y_train_dict["plate_x"],
    "plate_z": y_train_dict["plate_z"],
})
by_pitcher_pt = df_loc.groupby(["pitcher", "pitch_type"]).agg(
    plate_x=("plate_x", "mean"),
    plate_z=("plate_z", "mean"),
    n=("plate_x", "count"),
).reset_index()
pitcher_plate_means = {}
for _, row in by_pitcher_pt.iterrows():
    if row["n"] < MIN_PITCHES_FOR_MEANS:
        continue
    pid = str(int(row["pitcher"]))
    pt = row["pitch_type"]
    if pid not in pitcher_plate_means:
        pitcher_plate_means[pid] = {}
    pitcher_plate_means[pid][pt] = {"plate_x": float(row["plate_x"]), "plate_z": float(row["plate_z"])}
with open(_SAVED / "pitcher_plate_means.json", "w") as f:
    json.dump(pitcher_plate_means, f, indent=2)
print(f"Saved pitcher_plate_means.json ({len(pitcher_plate_means)} pitchers with >= {MIN_PITCHES_FOR_MEANS} pitches per type)")

Saved pitcher_plate_means.json (807 pitchers with >= 20 pitches per type)


In [18]:
#list the model and json files we wrote to saved_models
list(_SAVED.glob("pitcher_*.json"))

[PosixPath('/Users/javiermacias/Desktop/Final Year Project/Final_Year_Project/Models/Training/saved_models/pitcher_plate_x.json'),
 PosixPath('/Users/javiermacias/Desktop/Final Year Project/Final_Year_Project/Models/Training/saved_models/pitcher_pitch_type.json'),
 PosixPath('/Users/javiermacias/Desktop/Final Year Project/Final_Year_Project/Models/Training/saved_models/pitcher_plate_means.json'),
 PosixPath('/Users/javiermacias/Desktop/Final Year Project/Final_Year_Project/Models/Training/saved_models/pitcher_release_spin_rate.json'),
 PosixPath('/Users/javiermacias/Desktop/Final Year Project/Final_Year_Project/Models/Training/saved_models/pitcher_repertoire.json'),
 PosixPath('/Users/javiermacias/Desktop/Final Year Project/Final_Year_Project/Models/Training/saved_models/pitcher_plate_z.json'),
 PosixPath('/Users/javiermacias/Desktop/Final Year Project/Final_Year_Project/Models/Training/saved_models/pitcher_pitch_type_rates.json'),
 PosixPath('/Users/javiermacias/Desktop/Final Year Pro

In [19]:
#for each pitcher compute how often they threw each pitch type and save so the app can blend with model probs
import json
df_pt = pd.DataFrame({"pitcher": X_train["pitcher"], "pitch_type": y_train_dict["pitch_type"]})
rates = df_pt.groupby("pitcher")["pitch_type"].value_counts(normalize=True).unstack(fill_value=0.0)
pitcher_pitch_type_rates = {}
for pid in rates.index:
    d = rates.loc[pid].to_dict()
    pitcher_pitch_type_rates[str(int(pid))] = {str(k): float(v) for k, v in d.items()}
with open(_SAVED / "pitcher_pitch_type_rates.json", "w") as f:
    json.dump(pitcher_pitch_type_rates, f, indent=2)
print(f"Saved pitcher_pitch_type_rates.json ({len(pitcher_pitch_type_rates)} pitchers)")

Saved pitcher_pitch_type_rates.json (863 pitchers)
